# 04. Engineering & Internals

This document is a deep dive into the specific algorithms and components that power ConfigStream.

## 1. The Pareto Sort Algorithm

Most aggregators sort by Latency (Ping) alone. This is flawed because a proxy that pings 50ms but fails 50% of requests is useless. We use a multi-objective sorting algorithm.

### The Formula



In [ ]:
Score = (NormalizedLatency * 0.5) + (FailureRate * 0.3) + (Unstability * 0.2)



1.  **Normalized Latency (50% Weight)**:
    *   `RawLatency / 1000`. Capped at 1.0 (1 second).
    *   A 100ms proxy gets `0.1`. A 2000ms proxy gets `1.0`.

2.  **Failure Rate (30% Weight)**:
    *   Derived from `ProxyHistoryTracker`.
    *   `1.0 - (Successes / TotalChecks)`.
    *   A robust proxy (100% success) gets `0.0`. A flaky one (50% success) gets `0.5`.

3.  **Unstability / Jitter (20% Weight)**:
    *   Derived from `uptime_percentage` (as a proxy for stability over time).
    *   `1.0 - Uptime`.

**Result**: The sorting key minimizes the Score. A fast, reliable proxy appears at the top. A fast but broken proxy is pushed down.

## 2. Adaptive Timeout Logic

Fixed timeouts are inefficient.
*   **Too Short**: We drop valid but slow proxies (False Negatives).
*   **Too Long**: We waste minutes waiting for dead proxies.

### The `AdaptiveTimeout` Class
We track the average response time for every **Domain/IP**.

**Algorithm**:
1.  **Cold Start**: Default timeout is 10s.
2.  **Learning**: Every success records the latency.
3.  **Calculation**:


In [ ]:
    TargetTimeout = AvgLatency + (3 * StandardDeviation)


4.  **Bounds**: Min 3s, Max 20s.

**Effect**: If `us.server.com` usually responds in 200ms +/- 50ms, we set the timeout to ~350ms. If it hangs, we cut it instantly. This speeds up the pipeline by 40-60%.

## 3. The SingBoxTester

The `SingBoxTester` (`src/configstream/testers_core.py`) is the interface between Python and the testing engines.

*   **Logic Flow**:


In [ ]:
    if protocol in ["http", "socks"]:
        return self._test_direct(proxy)  # Uses aiohttp
    elif self.go_tester.available:
        return self.go_tester.test_batch(batch)  # Uses Go Sidecar
    else:
        return self._test_via_singbox(proxy)  # Uses Sing-box subprocess



### The Go Sidecar (Batch Tester)
*   **Path**: `src/go/tester/main.go`
*   **Concurrency**: Uses Go routines. Can handle 500 concurrent checks easily.
*   **Interface**: Reads JSON lines from STDIN, writes JSON lines to STDOUT.
*   **Honeypot Check**: Optionally performs a canary request using a `CANARY_URL` when `strict_security` is enabled.
    *   If `CANARY_URL` is **not** set, the tester logs a **warning** and disables honeypot detection while still measuring latency.
    *   This makes configuration drift visible without breaking the overall test pipeline.

## 4. Pipeline Orchestration & Backpressure

The `run_full_pipeline` function (`src/configstream/pipeline.py`) orchestrates producer/consumer stages, concurrency tuning, and output generation.

*   **Work Queue**: Uses an `asyncio.Queue` with a **max size of 500** to provide sufficient buffering between the fetcher and tester stages without risking deadlocks under high load.
*   **Timeouts**: The consumer side uses `asyncio.wait_for(..., timeout=300.0)` to avoid blocking indefinitely if the producer dies unexpectedly.
*   **Event Stream Lifecycle**: The `EventStream` is now closed in a `finally` block to guarantee that file handles and buffers are flushed even if the pipeline raises an exception.

## 4. Intelligence Layers

### Source Quality Tracker (`src/configstream/source_quality.py`)
Tracks the historical performance of every subscription source.

*   **Metrics**:
    *   **Reliability**: `working_proxies / fetched_proxies`
    *   **Consistency**: Variance in reliability over the last 10 runs.
    *   **Diversity**: How many unique ASNs (ISPs) does this source provide?
*   **Action**:
    *   **Gold**: High reliability + High diversity. Fetched first.
    *   **Silver**: Average. Fetched if capacity allows.
    *   **Garbage**: <1% reliability. Disabled automatically.

### Anomaly Detector (`src/configstream/anomaly.py`)
Detects "Pollution Attacks" or "Spam Batches."

1.  **Subnet Flood**:
    *   If a batch of 100 proxies contains 95 from `1.2.3.0/24`, it is suspicious.
    *   Action: Discard the whole batch.
2.  **Port Scanning**:
    *   If a batch contains sequential ports on the same IP (`1.2.3.4:1001`, `1.2.3.4:1002`...), we flag it.

## 5. Proxy Washing & Smart Chaining

### `ProxyWasher` (`src/configstream/intelligence/washer.py`)
*   **Problem**: Many IPs are "dirty" (blacklisted by Google/Cloudflare).
*   **Solution**: We wrap the proxy in a **Chain**.
    *   `Client -> DirtyProxy -> WARP (Clean IP) -> Target`
*   **Implementation**: We maintain a pool of valid Cloudflare WARP WireGuard keys. We generate a Sing-box "Chain" configuration that routes the outbound traffic of the dirty proxy into the WARP interface.
*   **Candidate Selection Update**: When a WARP pool is configured, we currently wash **all working proxies** (not just those tagged `dirty_ip`), providing a safer default in case tagging fails upstream.

### BoundedConcurrencyManager
To prevent "thundering herd" problems and OOM kills, we use a custom `BoundedConcurrencyManager`.
*   It uses an `asyncio.Condition` to manage a pool of permits.
*   It supports **Dynamic Resizing**: If the error rate spikes (indicating server overload or rate limiting), we dynamically reduce the concurrency limit. If errors drop, we increase it (AIMD - Additive Increase Multiplicative Decrease).

## 6. Static Vectors (Vector Search)

To enable "Natural Language Search" on a static site:
1.  **Vector Generation**: We convert proxy attributes (Country, City, ISP, Protocol, Speed Tag) into a simplistic low-dimensional vector, using SHA‑256–based feature hashing for consistency.
2.  **Pre-computation**: We generate `output/vectors.json` mapping `ProxyID -> [Vector]`.
3.  **Client-Side (Current)**: The frontend uses these vectors together with basic metadata to compute a simple keyword‑based relevance score (see `06-frontend.md`).
4.  **Client-Side (Planned)**: A future enhancement may use true cosine similarity over these vectors in a Web Worker to provide semantic search on large datasets.




---



# 05. DevOps & Infrastructure

ConfigStream relies on a robust, automated DevOps pipeline. We adhere to "GitOps" principles: everything is code, and the repository state is the source of truth.

## 1. The GitHub Actions Pipeline

The heart of the system is `.github/workflows/pipeline.yml`.

### Workflow Triggers
*   **Schedule**: `cron: "0 */6 * * *"` (Every 6 hours).
*   **Manual**: `workflow_dispatch` (For emergency updates).
*   **Push**: On changes to `main` (For testing code changes).

### Job Architecture

The pipeline uses a **Matrix Strategy** to parallelize work.

```mermaid
graph TD
    A[Setup Job] --> B{Matrix Jobs 1..6}
    B -->|Shard 1| C1[Run Batch 1]
    B -->|Shard 2| C2[Run Batch 2]
    B -->|...| C3[Run Batch 6]
    C1 --> D[Merge & Publish]
    C2 --> D
    C3 --> D
    D --> E[Deploy Pages]
```

#### 1. Setup (`setup`)
*   Installs dependencies (`pip`, `go`, `cargo`).
*   Restores caches:
    *   `pip-cache`
    *   `go-build-cache`
    *   `intelligence-db-cache` (`data/*.db`)
*   Pre-warms DNS cache.

#### 2. Sharding (`aggregator`)
*   We split `sources/` into 6 batch files (`sources/batch_1.txt` to `batch_6.txt`).
*   Each job in the matrix picks one batch file and processes it independently.
*   **Result**: Each job outputs a `partial_output_{id}.zip`.

#### 3. Merge (`merge_results`)
*   Downloads all partial artifacts.
*   Merges:
    *   Proxy lists (Deduplication).
    *   SQLite Databases (`anomaly.db`, `source_quality.db`) using `merge_from` logic.
*   Generates final `metadata.json` and `summary.json`.
*   Commits updated databases back to a persistent cache branch or artifact storage.

## 2. Caching Strategy

We heavily utilize `actions/cache` to persist state between ephemeral runners.

| Cache Key | Contents | Purpose |
| :--- | :--- | :--- |
| `db-sqlite-{hash}` | `data/*.db` | Persist Anomaly/Quality history. |
| `mmdb-geoip` | `data/*.mmdb` | Avoid redownloading GeoIP DBs. |
| `warp-keys` | `data/warp_keys.json` | Keep valid WARP identities. |

## 3. Security in CI/CD

*   **Secret Scanning**: We use `gitleaks` in pre-commit hooks to prevent committing API keys.
*   **Dependency Pinning**: All dependencies in `pyproject.toml` are pinned or version-ranged to prevent supply chain attacks.
*   **Minimum Permissions**: The `GITHUB_TOKEN` has read-only access except for the specific `deploy` job which needs write access to `gh-pages`.

## 4. Deployment & CDNs

### GitHub Pages
*   **Branch**: `gh-pages` (Orphan branch).
*   **Content**: The `output/` directory + `frontend/` assets.
*   **CDN**: Served via Fastly (GitHub's partner).

### Mirrors (Redundancy)
If GitHub is blocked, we automatically mirror to:
1.  **Cloudflare Pages**: Via a separate workflow trigger or pull-model.
2.  **IPFS**: We pin the output folder to IPFS using a pinning service (like Pinata) if configured.
3.  **Hugging Face**: We push datasets to Hugging Face Hub for ML usage.

## 5. Local Development

We provide a `docker-compose.yml` for replicating the CI environment.



In [ ]:
%%bash
# Start the pipeline locally
docker compose up --build

# Run the web server
docker compose up web



### Environment Variables
*   `MAX_WORKERS`: Control concurrency (Default: Auto).
*   `WARP_KEY_POOL`: JSON list of keys for the Washer.
*   `TELEGRAM_BOT_TOKEN`: For the bot interface.




---



# 08. API Reference

## CLI Reference

ConfigStream is primarily driven by its Command Line Interface (CLI).

### Global Options
*   `--help`: Show help message and exit.
*   `--version`: Show the version number.

### Commands

#### `run`
Executes the main aggregation pipeline.


In [ ]:
!configstream run --sources sources/batch_1.txt --output output/ --max-workers 50


**Options:**
*   `--sources`: Path to source file or URL (Required).
*   `--output`: Directory to save results (Default: `output/`).
*   `--max-workers`: Number of concurrent workers (Default: Auto).
*   `--timeout`: Connection timeout in seconds (Default: 10).
*   `--country`: Filter by country code (e.g., `IR`, `CN`).
*   `--strict`: Enable strict security checks (honeypot detection).

#### `serve`
Starts the API server.


In [ ]:
!configstream serve --host 0.0.0.0 --port 8000



#### `bot`
Starts the Telegram bot (polling mode).


In [ ]:
!configstream bot



#### `generate-warp`
Generates a Cloudflare WARP WireGuard configuration.


In [ ]:
!configstream generate-warp



## REST API (FastAPI)

The server exposes the following endpoints:

### Public Endpoints

#### `GET /api/stats`
Returns current pipeline statistics and last run status.
```json
{
  "last_updated": "2023-10-27T10:00:00Z",
  "total_proxies": 1500,
  "working_proxies": 1200,
  "sources_count": 50
}
```

#### `GET /api/convert`
Converts a subscription URL or content to a different format.
**Query Params:**
*   `url`: The subscription URL.
*   `target`: Target format (`clash`, `singbox`, `base64`).

#### `GET /health`
Health check endpoint for monitoring.
```json
{
  "status": "ok",
  "output_dir": "output/",
  "files_present": ["singbox.json", "clash.yaml"]
}
```

## Data Formats

### Metadata (`metadata.json`)
The `metadata.json` file contains summary data for the frontend.
```json
{
  "generated_at": "...",
  "stats": { ... },
  "proxies": [
    {
      "id": "...",
      "protocol": "vmess",
      "country": "US",
      "latency": 150,
      "reliability": 0.95
    }
  ]
}
```




---



# Configuration Reference

ConfigStream is configured via Environment Variables.

## Core Settings

| Variable | Default | Description |
| :--- | :--- | :--- |
| `MAX_WORKERS` | `50` | Concurrency for Python fetchers/parsers. |
| `TEST_TIMEOUT` | `10` | Timeout in seconds for proxy testing. |
| `CANARY_URL` | *None* | URL to check for Honeypot detection (e.g., a signed endpoint). |

## Intelligence Layer (v2.0)

| Variable | Default | Description |
| :--- | :--- | :--- |
| `WARP_KEY_POOL` | `[]` | JSON array of Cloudflare WARP credentials. **Required for Washing.** |
| `WARP_PORT` | `2408` | Target port for WARP endpoints (usually 2408 or 500-1000 range). |
| `RELAY_COUNTRY_CODE` | `IR` | Origin country for "Intranet Bridge" chains. |

## External Services

| Variable | Default | Description |
| :--- | :--- | :--- |
| `VT_API_KEY` | *None* | VirusTotal API key for IP reputation checks. |
| `TELEGRAM_BOT_TOKEN` | *None* | Bot token for uploading results to Telegram. |
| `TELEGRAM_CHAT_ID` | *None* | Chat ID for Telegram upload. |

## File Paths

*   `sources/batch_*.txt`: Input proxy sources.
*   `output/`: Generated artifacts (JSON, YAML, HTML).
*   `data/`: Persistent DBs (GeoIP, Quality Score).




---



# Development Guide

Welcome to ConfigStream v2.0 development!

## Environment Setup

1.  **Python:** Python 3.10+ required.


In [ ]:
!    pip install -e ".[dev]"


2.  **Go:** Go 1.21+ required for the tester.


In [ ]:
%%bash
    cd src/go/tester
    go mod tidy
    go build -o configstream-tester .


3.  **Frontend:** No build step required (Vanilla JS), but `playwright` is needed for testing.


In [ ]:
!    playwright install chromium



## Core Modules

### 1. `src/configstream/pipeline_core/`
The heart of the system.
-   `orchestrator.py`: Manages the flow.
-   `output_handler.py`: **NEW** Handles the Intelligence phase (Scan -> Wash -> Chain -> Write).

### 2. `src/configstream/intelligence/`
The "Brain" of v2.0.
-   `washer/core.py`: Wraps proxies in WARP.
-   `washer/chaining.py`: Generates topology-aware chains.
-   `vectors.py`: Generates AI feature vectors.

### 3. `src/go/tester/`
The "Muscle".
-   `main.go`: SOCKS5 Verifier.
-   `scanner/`: **NEW** High-performance UDP scanner for Cloudflare endpoints.

## Testing

Run unit tests:


In [ ]:
!pytest tests/unit



Run E2E tests (simulated):


In [ ]:
!pytest tests/e2e



**Note:** Some tests require the Go binary to be present in the path or project root.

## Contribution Workflow

1.  Create a feature branch.
2.  Implement changes.
3.  Run `black .`, `flake8 .`, `mypy .`
4.  Run `pytest`.
5.  Submit PR.




---



# 09. Contributor Guide

We welcome contributions! This guide will help you get started.

## Development Environment

### Prerequisites
*   Python 3.10+
*   Go 1.21+ (for the Go Tester sidecar)
*   Docker (optional, for local containerized runs)

### Setup
1.  **Clone the repository:**


In [ ]:
%%bash
    git clone https://github.com/your-repo/configstream.git
    cd configstream



2.  **Create a virtual environment:**


In [ ]:
%%bash
    python -m venv venv
    source venv/bin/activate



3.  **Install dependencies:**


In [ ]:
!    pip install -e ".[dev]"


    This installs the package in editable mode along with development tools (flake8, mypy, pytest, etc.).

4.  **Install Playwright browsers (for E2E tests):**


In [ ]:
!    playwright install --with-deps



## Coding Standards

We enforce strict code quality standards.

*   **Formatting:** We use `black`.


In [ ]:
!    black .


*   **Linting:** We use `flake8`.


In [ ]:
!    flake8 src tests


*   **Type Checking:** We use `mypy`.


In [ ]:
!    mypy src



## Testing

*   **Run all tests:**


In [ ]:
!    pytest


*   **Run unit tests only:**


In [ ]:
!    pytest tests/unit


*   **Run E2E tests:**


In [ ]:
!    pytest tests/e2e



**Coverage Requirement:** We aim for >90% test coverage. Please write tests for any new features or bug fixes.

## Pull Request Workflow

1.  Create a new branch for your feature or fix.
2.  Write code and tests.
3.  Ensure all checks pass locally (`pytest`, `flake8`, `mypy`).
4.  Submit a Pull Request with a clear description of your changes.
5.  Address any review comments.

## Project Structure
*   `src/configstream`: Main Python package.
*   `src/go`: Go-based high-performance components.
*   `frontend`: Web interface assets.
*   `tests`: Test suite.
*   `docs/wiki`: Documentation.

